# TabPFN for Decision Support
### A comparative study on small tabular classification data

This project explores the use of TabPFN for data-driven decision support on a small structured dataset.

Using the Wisconsin Breast Cancer dataset, I compare three classification approaches:

- Logistic Regression
- Random Forest
- TabPFN

The analysis focuses not only on overall predictive performance, but also on the detection of malignant cases and on cases where the models disagree.

## Research Question

**How does TabPFN compare with established machine-learning baselines on a small tabular classification task, and what can we learn from the models' misclassifications?**

> This project is an exploratory machine-learning benchmark and is not intended for clinical use.

## Setup

The notebook uses the lightweight `tabpfn-client` for TabPFN inference. In Google Colab, the remaining data-science packages are already available.

In [ ]:
!pip install -q tabpfn-client

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    recall_score,
)

from tabpfn_client import TabPFNClassifier

## 1. Data

The analysis uses the Wisconsin Diagnostic Breast Cancer dataset provided by scikit-learn.

The dataset contains 569 observations and 30 numerical features derived from digitized images of breast mass cell nuclei.

**Target classes**
- `0` = malignant
- `1` = benign

A stratified 75/25 train-test split is used to preserve the class distribution.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Dataset:", X.shape)
print("Training:", X_train.shape)
print("Test:", X_test.shape)
print("Target classes:", dict(enumerate(data.target_names)))

## 2. Baseline Models

Two established machine-learning approaches are used as baselines.

### Logistic Regression
A standardized Logistic Regression provides a simple and interpretable linear benchmark.

### Random Forest
A Random Forest provides a non-linear ensemble-based benchmark.

Both models are evaluated on exactly the same train-test split as TabPFN.

In [ ]:
# Logistic Regression
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000),
)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

# Random Forest
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
)
random_forest.fit(X_train, y_train)
rf_predictions = random_forest.predict(X_test)
rf_probabilities = random_forest.predict_proba(X_test)[:, 1]

print("Baseline models successfully trained.")

## 3. TabPFN

TabPFN is evaluated on the same training and test data as the baseline models.

The experiment uses the TabPFN client to perform inference while keeping the experimental setup identical across all three approaches.

In [ ]:
tabpfn_model = TabPFNClassifier()
tabpfn_model.fit(X_train, y_train)

tabpfn_predictions = tabpfn_model.predict(X_test)
tabpfn_probabilities = tabpfn_model.predict_proba(X_test)[:, 1]

print("TabPFN inference completed.")

## 4. Model Evaluation

The models are compared using four metrics:

- **Accuracy** – overall proportion of correct classifications
- **F1-score** – balance between precision and recall for the positive class
- **ROC-AUC** – ability to distinguish between the two classes across classification thresholds
- **Malignant Recall** – proportion of malignant cases correctly identified

Malignant recall is included because aggregate performance metrics alone may hide errors in the class that is particularly important in this example.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "TabPFN",
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, tabpfn_predictions),
    ],
    "F1-Score": [
        f1_score(y_test, logistic_predictions),
        f1_score(y_test, rf_predictions),
        f1_score(y_test, tabpfn_predictions),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, logistic_probabilities),
        roc_auc_score(y_test, rf_probabilities),
        roc_auc_score(y_test, tabpfn_probabilities),
    ],
    "Malignant Recall": [
        recall_score(y_test, logistic_predictions, pos_label=0),
        recall_score(y_test, rf_predictions, pos_label=0),
        recall_score(y_test, tabpfn_predictions, pos_label=0),
    ],
})

results.round(3)

In [ ]:
results_plot = results.set_index("Model")

ax = results_plot.plot(
    kind="bar",
    figsize=(10, 6),
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("")
plt.ylim(0.90, 1.00)
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

### Results

Logistic Regression achieved the strongest performance on this particular train-test split, with an accuracy of 0.986 and a ROC-AUC of 0.998.

TabPFN performed very closely, reaching an accuracy of 0.979 and a ROC-AUC of 0.997, while outperforming the Random Forest baseline in accuracy, F1-score, and malignant recall.

The results show that increased model complexity does not automatically lead to better predictive performance. They also highlight the importance of evaluating different modelling approaches empirically rather than assuming that one method will perform best across datasets.

## 5. Error Analysis

Aggregate performance metrics do not reveal whether different models fail on the same observations.

To investigate this, predictions from all three models are compared at the individual test-case level. Particular attention is given to malignant cases that were incorrectly classified as benign.

In [ ]:
comparison = X_test.copy()
comparison["Actual"] = y_test
comparison["Logistic Regression"] = logistic_predictions
comparison["Random Forest"] = rf_predictions
comparison["TabPFN"] = tabpfn_predictions

errors = comparison[
    (comparison["Logistic Regression"] != comparison["Actual"])
    | (comparison["Random Forest"] != comparison["Actual"])
    | (comparison["TabPFN"] != comparison["Actual"])
]

print("Test cases with at least one misclassification:", len(errors))
errors[["Actual", "Logistic Regression", "Random Forest", "TabPFN"]]

### Misclassification Patterns

Seven test cases were misclassified by at least one model.

Among the malignant cases:

- Logistic Regression missed 1 case.
- TabPFN missed 2 cases.
- Random Forest missed 4 cases.
- One malignant case (case 73) was misclassified by all three models.

The shared misclassification of case 73 motivated a closer exploratory examination of this observation.

In [ ]:
missed_malignant = comparison[
    (comparison["Actual"] == 0)
    & (
        (comparison["Logistic Regression"] != 0)
        | (comparison["Random Forest"] != 0)
        | (comparison["TabPFN"] != 0)
    )
]

missed_malignant[["Actual", "Logistic Regression", "Random Forest", "TabPFN"]]

### Exploring a Shared Misclassification

Case 73 was malignant but was classified as benign by all three models.

To explore why this observation may have been particularly challenging, selected features are compared with the average values of the malignant and benign groups.

This descriptive comparison provides context for the shared misclassification but should not be interpreted as a formal explanation of the models' predictions.

In [ ]:
selected_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean concavity",
    "mean concave points",
]

case_comparison = pd.DataFrame({
    "Case 73": X.loc[73, selected_features],
    "Malignant mean": X[y == 0][selected_features].mean(),
    "Benign mean": X[y == 1][selected_features].mean(),
})

case_comparison.round(3)

#### Interpretation

Several characteristics of case 73 differ substantially from the average malignant observation.

For example, its mean radius (13.80) and mean area (584.10) are considerably lower than the malignant group averages (17.46 and 978.38) and lie closer to the corresponding benign averages (12.15 and 462.79).

Mean texture is even lower than the benign group average in this comparison.

The selected features therefore suggest that case 73 exhibits characteristics that partially resemble the benign group. This may have contributed to the difficulty experienced by all three classifiers.

However, this is an exploratory comparison of selected features and group averages, not a causal or model-specific explanation.

## 6. Conclusion

This project explored how TabPFN performs on a small structured classification task compared with two established machine-learning baselines.

On this particular train-test split, Logistic Regression achieved the strongest overall performance, with an accuracy of 0.986 and a ROC-AUC of 0.998. TabPFN performed very closely, reaching an accuracy of 0.979 and a ROC-AUC of 0.997, while outperforming the Random Forest baseline in accuracy, F1-score, and malignant recall.

The experiment demonstrates that model complexity alone does not determine predictive performance. A comparatively simple, well-specified baseline can remain highly competitive, even when compared with more advanced approaches.

The error analysis provided an additional perspective beyond aggregate performance metrics. Only seven test observations were misclassified by at least one model, and one malignant case was incorrectly classified as benign by all three approaches. An exploratory examination showed that several selected features of this observation were closer to the benign group averages, illustrating the value of examining individual prediction errors alongside overall model scores.

### Key Takeaways

- TabPFN achieved strong predictive performance with minimal model-specific setup.
- Logistic Regression provided the strongest results on this particular train-test split.
- TabPFN outperformed the Random Forest baseline on accuracy, F1-score, and malignant recall.
- Model complexity alone did not determine predictive performance.
- Class-specific evaluation provided additional insight beyond overall accuracy.
- Error analysis revealed observations that were challenging across different modelling approaches.
- Model comparison should remain empirical and context-dependent.

### Limitations

This project is an exploratory benchmark based on a small public dataset and a single stratified train-test split.

The results should therefore not be interpreted as evidence that one modelling approach is generally superior to another. Performance may vary with different data splits, datasets, preprocessing choices, hyperparameters, and evaluation strategies.

The examination of case 73 is descriptive and based on selected features and group averages. It is not a formal model-interpretability analysis and does not establish why the individual models produced their predictions.

The dataset is used solely as a public machine-learning benchmark. This project is not intended for clinical use or medical decision-making.

### Possible Next Steps

Future extensions could include:

- repeated stratified cross-validation,
- evaluation on additional tabular datasets,
- probability calibration analysis,
- comparison of training and inference workflows,
- systematic model interpretability methods,
- and investigation of TabPFN on different types of structured decision-support problems.